<a href="https://colab.research.google.com/github/cloudmrhub/camrie-tools/blob/v1/camrie_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CAMRIE Tools — Colab Guide

This notebook installs the full `camrie-tools` stack, checks the installation, and runs a small functionality example.

The repository/package follows the CAMRIE app family naming convention:

- `camrie` — main app
- `camrie-webgui` — frontend
- `camrie-tools` — Python package distribution
- `camrie-app` — cloud builder

Python imports use `camrie_tools` because Python module names cannot contain hyphens.

Use a GPU runtime if you want GPU execution. The package can still install on CPU-only runtimes; `CUDA.functional()` reports whether GPU execution is actually available.

## 1. Install Julia

Colab Python runtimes usually do not include Julia, so install it first.

In [ ]:
!curl -fsSL https://install.julialang.org | sh -s -- -y

In [ ]:
import os

os.environ["PATH"] = f"{os.path.expanduser('~')}/.juliaup/bin:" + os.environ["PATH"]
!julia --version

## 2. Install camrie-tools

In [ ]:
%pip install git+https://github.com/cloudmrhub/camrie-tools@v1

In [ ]:
import camrie_tools

print("camrie_tools", camrie_tools.__version__)
print("Bundled Julia script:", camrie_tools.simulate_batch_path())

## 3. Install the full Julia dependency stack

This installs `KomaInterface.jl` and `CUDA.jl`. HTTPS is used here because Colab does not have your GitHub SSH keys unless you configure them manually.

In [ ]:
!camrie-install-julia

## 4. Verify, run examples, and show a reconstruction

In [ ]:
!camrie-test-installation

In [ ]:
!camrie-example

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from camrie_tools._reconstruction_smoke import run_reconstruction_smoke

output_dir = Path("/content/camrie_reconstruction_smoke")
if not output_dir.parent.exists():
    output_dir = Path("camrie_reconstruction_smoke")

summary = run_reconstruction_smoke(output_dir=str(output_dir), n_threads=1)
print(json.dumps({
    "output_dir": str(output_dir),
    "spin_count": summary["spin_count"],
    "kspace_shape": summary["kspace_shape"],
    "reconstruction_shape": summary["reconstruction_shape"],
    "peak": summary["peak"],
}, indent=2))

recon = np.load(summary["outputs"]["reconstruction_magnitude"])

plt.figure(figsize=(5, 5))
plt.imshow(recon, cmap="gray")
plt.title("CAMRIE circular phantom reconstruction")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# Import the pipeline module when the runtime dependencies are installed.
from camrie_tools.MRI_pipeline import read_pulseq_params

print(read_pulseq_params)